In [1]:
import pickle 

with open(
    "../../data/processed/CTB/prosit_params/sim_log_s6_sample_30.000_depth10_params_HO2_LL_2.pkl",
    "rb"
) as f:

    baseline_params = pickle.load(f)

for act in sorted(
    baseline_params.act_to_resources.keys()
):
    print(act)
print(baseline_params.act_to_resources)

Gate In
Gate Out
HO2_delivery
HO2_mixed
HO2_receive
LL_delivery
LL_mixed
LL_receive
RMG_delivery
RMG_mixed
RMG_receive
{'HO2_mixed': ['HO2'], 'LL_mixed': ['LL'], 'RMG_delivery': ['T14', 'T13', 'T12', 'T17', 'T16', 'T15', 'T21', 'T10', 'T24', 'T11', 'T09', 'T20', 'T19', 'T18', 'T06', 'T27', 'T23', 'T22', 'T07', 'T08', 'T25', 'T26'], 'HO2_receive': ['HO2'], 'RMG_mixed': ['T13', 'T24', 'T14', 'T16', 'T23', 'T20', 'T22', 'T21', 'T27', 'T17', 'T18', 'T10', 'T15', 'T06', 'T09', 'T11', 'T12', 'T07', 'T08', 'T26', 'T19', 'T25'], 'LL_receive': ['LL'], 'Gate Out': ['Res.GateOut'], 'LL_delivery': ['LL'], 'HO2_delivery': ['HO2'], 'RMG_receive': ['T13', 'T24', 'T14', 'T16', 'T11', 'T20', 'T27', 'T15', 'T21', 'T18', 'T22', 'T23', 'T17', 'T12', 'T06', 'T10', 'T26', 'T09', 'T08', 'T07', 'T19', 'T25'], 'Gate In': ['Res.GateIn']}


In [2]:
import pickle 

with open(
    "../../data/processed/CTB/prosit_simulations/scenario_a/scenario_A_T20_closed.pkl",
    "rb"
) as f:

    baseline_params = pickle.load(f)

for act in sorted(
    baseline_params.act_to_resources.keys()
):
    print(act)
print(baseline_params.act_to_resources)

Gate In
Gate Out
HO2_delivery
HO2_mixed
HO2_receive
LL_delivery
LL_mixed
LL_receive
RMG_delivery
RMG_mixed
RMG_receive
{'HO2_mixed': ['HO2'], 'LL_mixed': ['LL'], 'RMG_delivery': ['T14', 'T13', 'T12', 'T17', 'T16', 'T15', 'T21', 'T10', 'T24', 'T11', 'T09', 'T19', 'T18', 'T06', 'T27', 'T23', 'T22', 'T07', 'T08', 'T25', 'T26'], 'HO2_receive': ['HO2'], 'RMG_mixed': ['T13', 'T24', 'T14', 'T16', 'T23', 'T22', 'T21', 'T27', 'T17', 'T18', 'T10', 'T15', 'T06', 'T09', 'T11', 'T12', 'T07', 'T08', 'T26', 'T19', 'T25'], 'LL_receive': ['LL'], 'Gate Out': ['Res.GateOut'], 'LL_delivery': ['LL'], 'HO2_delivery': ['HO2'], 'RMG_receive': ['T13', 'T24', 'T14', 'T16', 'T11', 'T27', 'T15', 'T21', 'T18', 'T22', 'T23', 'T17', 'T12', 'T06', 'T10', 'T26', 'T09', 'T08', 'T07', 'T19', 'T25'], 'Gate In': ['Res.GateIn']}


In [8]:
import pandas as pd

baseline_sim = pd.read_csv("../../data/processed/CTB/prosit_simulations/sim_log_s6_sample_24.000_depth10_maxConc_2.csv")
scenario_sim = pd.read_csv("../../data/processed/CTB/prosit_simulations/scenario_a/sim_log_24.000_T20_closed.csv")

scenario = scenario_sim
baseline = baseline_sim
ref = baseline_sim

In [ ]:
# =============CALCULATE SCENARIO KPIS============================================
# CALCULATE SCENARIO KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    scenario[col] = pd.to_datetime(
        scenario[col]
    )

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

scenario["waiting_time"] = (
    scenario["start:timestamp"]
    - scenario["enabled:timestamp"]
).dt.total_seconds() / 60

scenario["service_time"] = (
    scenario["time:timestamp"]
    - scenario["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

case_start = (
    scenario.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

case_end = (
    scenario.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

scenario["turnaround_time"] = (
    case_end
    - case_start
).dt.total_seconds() / 60

In [10]:
#========== CALCULTATE REF KPIS =================
# ==========================================================
# CALCULATE REFERENCE KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    ref[col] = pd.to_datetime(ref[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

ref["waiting_time"] = (
    ref["start:timestamp"]
    - ref["enabled:timestamp"]
).dt.total_seconds() / 60

ref["service_time"] = (
    ref["time:timestamp"]
    - ref["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

ref_case_start = (
    ref.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

ref_case_end = (
    ref.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

ref["turnaround_time"] = (
    ref_case_end
    - ref_case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Reference cases: "
    f"{ref['case:concept:name'].nunique():,}"
)

print(
    "\nKPI columns created:"
)

print(
    [
        "waiting_time",
        "service_time",
        "turnaround_time"
    ]
)

Reference RMG events: 25,205
Reference cases: 24,000

KPI columns created:
['waiting_time', 'service_time', 'turnaround_time']


In [4]:
rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

scenario_rmg = scenario[
    scenario["concept:name"]
    .isin(rmg_activities)
].copy()

In [11]:
def compare_scenarios(
    baseline_values,
    scenario_values,
    kpi,
    segment
):

    baseline_values = (
        pd.Series(baseline_values)
        .dropna()
    )

    scenario_values = (
        pd.Series(scenario_values)
        .dropna()
    )

    return {

        "segment": segment,
        "kpi": kpi,

        "baseline_mean":
        baseline_values.mean(),

        "scenario_mean":
        scenario_values.mean(),

        "baseline_median":
        baseline_values.median(),

        "scenario_median":
        scenario_values.median(),

        "change_pct":
        100
        * (
            scenario_values.mean()
            - baseline_values.mean()
        )
        / baseline_values.mean()
    }

In [12]:
baseline_rmg = baseline[
    baseline["concept:name"]
    .isin(rmg_activities)
].copy()

results = []

segments = {

    "receive": (
        baseline_rmg[
            baseline_rmg["concept:name"]
            == "RMG_receive"
        ],

        scenario_rmg[
            scenario_rmg["concept:name"]
            == "RMG_receive"
        ]
    ),

    "delivery": (
        baseline_rmg[
            baseline_rmg["concept:name"]
            == "RMG_delivery"
        ],

        scenario_rmg[
            scenario_rmg["concept:name"]
            == "RMG_delivery"
        ]
    ),

    "mixed": (
        baseline_rmg[
            baseline_rmg["concept:name"]
            == "RMG_mixed"
        ],

        scenario_rmg[
            scenario_rmg["concept:name"]
            == "RMG_mixed"
        ]
    )
}

for segment, (
    base_seg,
    scen_seg
) in segments.items():

    for kpi in [
        "waiting_time",
        "service_time"
    ]:

        results.append(

            compare_scenarios(
                base_seg[kpi],
                scen_seg[kpi],
                kpi,
                segment
            )

        )

scenario_kpis = pd.DataFrame(
    results
)

scenario_kpis.round(2)

,segment,kpi,baseline_mean,scenario_mean,baseline_median,scenario_median,change_pct
0,receive,waiting_time,27.80,30.13,6.0,6.0,8.36
1,receive,service_time,16.72,15.73,9.0,9.0,-5.93
2,delivery,waiting_time,27.86,31.85,6.0,7.0,14.34
3,delivery,service_time,12.09,10.93,7.0,7.0,-9.53
4,mixed,waiting_time,24.18,32.07,6.0,7.0,32.62
5,mixed,service_time,18.00,18.48,11.0,10.0,2.66


In [13]:
baseline_cases = (
    baseline.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

scenario_cases = (
    scenario.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

turnaround_compare = pd.DataFrame(
    [
        compare_scenarios(
            baseline_cases,
            scenario_cases,
            "turnaround_time",
            "case_level"
        )
    ]
)

turnaround_compare.round(2)

,segment,kpi,baseline_mean,scenario_mean,baseline_median,scenario_median,change_pct
0,case_level,turnaround_time,55.96,57.48,28.0,28.0,2.71
